# Feature extraction with BERT

In [2]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
import numpy as np
from tqdm import tqdm

/Users/jordan/Desktop/Live_Projects/FYP_Documents/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")
model.to(device)
model.eval()

def extract_bert_features_batch(texts, batch_size=16, max_length=512):
    """
    Extract BERT embeddings in batches for efficiency
    """
    all_features = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize batch
        encoding = tokenizer.batch_encode_plus(
            batch_texts,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        # Move to device
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        
        # Extract features
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # Get [CLS] token embeddings
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        x
        all_features.append(cls_embeddings)
    
    return np.vstack(all_features)


Using device: mps


In [6]:
df = pd.read_csv("processed_datasets/WELFake_Dataset_processed.csv")

# Drop unnecessary columns
df = df[['combined_text', 'label']]

# Extract features and save incrementally
print("Extracting BERT features in batches and saving incrementally...")

batch_size = 16
texts = df['combined_text'].tolist()
labels = df['label'].values

# Create CSV file with headers
first_batch = True

for i in tqdm(range(0, len(texts), batch_size), desc="Processing batches"):
    batch_texts = texts[i:i + batch_size]
    batch_labels = labels[i:i + batch_size]
    
    # Tokenize batch
    encoding = tokenizer.batch_encode_plus(
        batch_texts,
        add_special_tokens=True,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Extract features
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    
    batch_df = pd.DataFrame(cls_embeddings, columns=[f'feature_{j}' for j in range(cls_embeddings.shape[1])])
    batch_df['label'] = batch_labels
    
    # Add to CSV
    batch_df.to_csv('preprocessed_datasets/bert_features_with_labels.csv', 
                    mode='w' if first_batch else 'a',
                    header=first_batch,
                    index=False)
    
    first_batch = False

print("Feature extraction complete! CSV saved incrementally.")

Extracting BERT features in batches and saving incrementally...


Processing batches:  20%|█▉        | 896/4508 [28:58<1:56:49,  1.94s/it]


KeyboardInterrupt: 